# Loan Analysis - Model Building

**Model:** Artificial Neural Network (ANN)

Primary metric: ROC-AUC

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay,
    precision_score, recall_score, f1_score
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
# Load processed data
processed = np.load("../data/processed/02_processed_data.npz")
X_train = processed['X_train']
X_test = processed['X_test']
y_train = processed['y_train']
y_test = processed['y_test']

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (887550, 1010), y_train: (887550,)
X_test: (443547, 1010), y_test: (443547,)


In [3]:
def print_score(true, pred, train=True):
    clf_report = pd.DataFrame(classification_report(true, pred, output_dict=True))
    label = "Train" if train else "Test"
    print(f"{label} Result:")
    print("=" * 50)
    print(f"Accuracy Score: {accuracy_score(true, pred) * 100:.2f}%")
    print("_" * 50)
    print(f"CLASSIFICATION REPORT:\n{clf_report}")
    print("_" * 50)
    print(f"Confusion Matrix:\n{confusion_matrix(true, pred)}\n")

## Artificial Neural Networks (ANNs)

In [4]:
def plot_learning_evolution(r):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(r.history['loss'], label='Loss')
    axes[0].plot(r.history['val_loss'], label='val_Loss')
    axes[0].set_title('Loss Evolution During Training')
    axes[0].legend()

    axes[1].plot(r.history['AUC'], label='AUC')
    axes[1].plot(r.history['val_AUC'], label='val_AUC')
    axes[1].set_title('AUC Score Evolution During Training')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def nn_model(num_columns, num_labels, hidden_units, dropout_rates, learning_rate):
    inp = Input(shape=(num_columns,))
    x = BatchNormalization()(inp)
    x = Dropout(dropout_rates[0])(x)
    for i in range(len(hidden_units)):
        x = Dense(hidden_units[i], activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout_rates[i + 1])(x)
    x = Dense(num_labels, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=x)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=[AUC(name='AUC')])
    return model

In [ ]:
num_columns = X_train.shape[1]
num_labels = 1
hidden_units = [150, 150, 150]
dropout_rates = [0.1, 0, 0.1, 0]
learning_rate = 1e-3

ann_model = nn_model(
    num_columns=num_columns,
    num_labels=num_labels,
    hidden_units=hidden_units,
    dropout_rates=dropout_rates,
    learning_rate=learning_rate
)

callbacks = [
    EarlyStopping(monitor='val_AUC', patience=5, restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

r = ann_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=2048,
    callbacks=callbacks
)

In [ ]:
plot_learning_evolution(r)

In [ ]:
y_train_pred = ann_model.predict(X_train)
print_score(y_train, y_train_pred.round(), train=True)

In [ ]:
y_test_pred = ann_model.predict(X_test)
print_score(y_test, y_test_pred.round(), train=False)

In [ ]:
train_probs = ann_model.predict(X_train).ravel()
test_probs = ann_model.predict(X_test).ravel()

scores_dict = {
    'ANNs': {
        'Train': roc_auc_score(y_train, train_probs),
        'Test': roc_auc_score(y_test, test_probs),
    },
}
print(f"ANN ROC-AUC - Train: {scores_dict['ANNs']['Train']:.4f}, Test: {scores_dict['ANNs']['Test']:.4f}")

In [ ]:
# Save model
os.makedirs('../models', exist_ok=True)
ann_model.save('../models/ann_model.keras')
print("Model saved to ../models/ann_model.keras")

In [ ]:
# Performance metrics CSV
prob_train = ann_model.predict(X_train).ravel()
prob_test = ann_model.predict(X_test).ravel()
pred_test = (prob_test >= 0.5).astype(int)

perf = pd.DataFrame([{
    'model': 'ANN',
    'roc_auc_train': roc_auc_score(y_train, prob_train),
    'roc_auc_test': roc_auc_score(y_test, prob_test),
    'accuracy_test': accuracy_score(y_test, pred_test),
    'precision_test': precision_score(y_test, pred_test, pos_label=0),
    'recall_test': recall_score(y_test, pred_test, pos_label=0),
    'f1_test': f1_score(y_test, pred_test, pos_label=0),
}]).set_index('model')

perf.to_csv('../data/processed/03_model_performance.csv')
print(perf.to_string())